# MIMIC-III -- ICU Length-of-Stay Prediction
## A CRISP-DM walkthrough

Predict a patient's **ICU length of stay (LOS, in days)** from the **first 24 hours**
of an ICU stay, using bedside vitals (CHARTEVENTS) and laboratory results (LABEVENTS)
from MIMIC-III. All logic lives in the `src/` package; this notebook orchestrates it
following the six **CRISP-DM** phases:

1. **Business Understanding** -- the question, its clinical value, target & window choice.
2. **Data Understanding** -- EDA: distributions, missingness, per-patient trajectories.
3. **Data Preparation** -- leak-free feature engineering, inventory, structure exploration
   (PCA / t-SNE / clustering), the train/test split.
4. **Modeling** -- literature baselines, model zoo, SOTA discussion, the LABEVENTS ablation.
5. **Evaluation** -- metrics vs trivial *and* published baselines, error & importance analysis.
6. **Deployment** -- profiling, big-data (MapReduce) & multiprocessing discussion, limitations.

> Methodology reference: Wirth & Hipp (2000), *CRISP-DM 1.0*. The phase headings below map
> directly onto it.

## Phase 1 -- Business Understanding

**Goal.** Estimate how long an ICU patient will stay, early enough to be useful for bed
management, staffing and step-down planning. A model that needs the whole stay to predict the
stay is worthless at deployment, so we predict **at the 24-hour mark**.

**Target (why this one).** ICU LOS in days = `OUTTIME - INTIME` (ICUSTAYS). We chose the
*ICU* stay (not the hospital admission) because: (a) it matches the chart-event data we model;
(b) it is the canonical benchmark target (Harutyunyan et al. 2019); (c) one `SUBJECT_ID` can
have several stays, which we handle by grouped splitting rather than by switching target.

**Window (why 24h).** A full day of vitals/labs exists, yet there is still time to act. A 6h
window has too few labs resulted; a 72h window delays every decision and excludes more short
stays. The cohort is therefore **stays still in the ICU at 24h** (`MIN_LOS_HOURS = 24`).

**Two framings (why both).** Regression (days) is the literal ask but is intrinsically
low-R^2; ordinal classification (short<3d / medium 3-7d / long>7d) is the more robust,
standard framing and gives clearer operational buckets. We report both.

**Why these feature categories.** First-24h **vitals** (haemodynamics, respiration, neuro),
**labs** (renal, electrolytes, perfusion, haematology, coagulation), **demographics/admin**
(age, sex, admission type, care unit) -- the variables a clinician actually has at hour 24 and
that the literature finds predictive of deterioration and stay length.

## Phase 2 -- Data Understanding

### 2.0 Environment & configuration

In [ ]:
import logging, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.base import clone

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
sys.path.insert(0, str(Path.cwd()))

from src import config as cfg
from src.data.loader import load_cohort
from src.data import sql, synthetic, concepts as vital_concepts, lab_concepts
from src.features.engineering import build_feature_matrix, categorize_features
from src.data.splits import grouped_train_test_split, assert_no_group_leakage
from src.models import registry
from src.evaluation import harness, metrics
from src.evaluation.baselines import literature_baselines
from src.evaluation.profiling import Profiler
from src.analysis import unsupervised as uns
from src.visualization import plots

USE_BIGQUERY = True     # False -> labelled synthetic fallback (runs anywhere)
profiler = Profiler()
print("window =", cfg.PREDICTION_WINDOW_HOURS, "h | include_labs =", cfg.INCLUDE_LABS)
print("credentials:", cfg.GCP_CREDENTIALS_PATH or "none (synthetic)")

### 2.1 Load data -- BigQuery first, synthetic fallback
Aggregation is pushed into BigQuery (see Phase 6). Only compact per-(stay, concept) tables
come back. LABEVENTS has no `ICUSTAY_ID`, so it joins on `HADM_ID`, time-windowed to `INTIME`.

In [ ]:
with profiler.track("1_load_data"):
    data = load_cohort(use_bigquery=USE_BIGQUERY,
                       window_hours=cfg.PREDICTION_WINDOW_HOURS,
                       limit=cfg.DEV_ICUSTAY_LIMIT, include_labs=True)
banner = "REAL MIMIC-III via BigQuery" if data.is_real else \
    "(!) SYNTHETIC DATA -- DEMONSTRATION ONLY, not real findings (!)"
print("=" * 72, f"\nDATA SOURCE: {data.source}  ({banner})\n", "=" * 72, sep="")
print("cohort       :", data.cohort.shape)
print("vital aggs   :", data.aggregates.shape)
print("lab aggs     :", data.lab_aggregates.shape)
print("demographics :", data.demographics.shape)
data.cohort.head()

### 2.2 Target distribution & class balance

In [ ]:
y_days = data.cohort["los_icu_days"].clip(upper=cfg.MAX_LOS_DAYS)
fig = plots.los_distribution(y_days); plots.save(fig, "los_distribution.png"); plt.show()
print(y_days.describe().round(2).to_string())
buckets = pd.cut(y_days, [0, cfg.LOS_SHORT_DAYS, cfg.LOS_MEDIUM_DAYS, np.inf],
                 labels=list(cfg.LOS_CLASS_LABELS), include_lowest=True)
print("\nclass balance:", buckets.value_counts().to_dict())
print("Right-skewed: most stays short, long heterogeneous tail -> regression is hard, the "
      "'long' class is the minority. Motivates the bucketed framing + balanced class weights.")

### 2.3 Per-patient view (the assignment's example plot + a population band)
Left idea: every measurement of one stay over time, coloured by concept. We also show a single
concept as a **line vs the cohort mean +/- 1 SD band** -- is this patient running high/low?

In [ ]:
example_id = int(data.cohort["icustay_id"].iloc[0])
if data.is_real:
    from src.data.loader import BigQueryClient
    tl = BigQueryClient().run(sql.patient_timeline_query(example_id, window_hours=48),
                              f"timeline_{example_id}")
else:
    tl = synthetic.patient_timeline(example_id, window_hours=48)
fig = plots.patient_timeline(tl, example_id); plots.save(fig, f"timeline_{example_id}.png"); plt.show()

concept = "heart_rate"
pm = data.aggregates.loc[data.aggregates.concept == concept, "mean"]
fig = plots.patient_vs_population(tl, concept, float(pm.mean()), float(pm.std()), example_id)
plots.save(fig, f"patient_band_{example_id}.png"); plt.show()

### 2.4 Missing-value analysis (and why missingness is *informative*)
CHARTEVENTS/LABEVENTS are ragged: not every concept is charted for every stay. Critically,
**missingness is not random** -- ordering a lactate or an extra vital is itself a clinical
decision driven by concern. So beyond imputing, in Phase 3 we add a 0/1 *was-measured* feature
per concept. Below: fraction of stays for which each concept is absent in the first 24h.

In [ ]:
n_total = data.cohort["icustay_id"].nunique()
def presence(agg, prefix=""):
    p = agg.groupby("concept")["icustay_id"].nunique() / n_total
    p.index = [f"{prefix}{c}" for c in p.index]
    return 1 - p   # -> missing fraction
miss = pd.concat([presence(data.aggregates), presence(data.lab_aggregates, "lab_")])
fig = plots.missingness_bar(miss.sort_values(ascending=False), top=len(miss))
plots.save(fig, "missingness.png"); plt.show()
print("Labs are missing far more often than vitals -- exactly why the 'was-measured' "
      "indicators (Phase 3) and a lenient lab missing-threshold matter.")

## Phase 3 -- Data Preparation

### 3.1 Leakage controls -- what is hidden or altered
The single biggest risk in LOS prediction is leaking the answer. Explicitly:

| Variable / source | Action | Why |
|---|---|---|
| `OUTTIME`, `DISCHTIME` | **hidden** (only define the target) | knowing discharge time *is* the answer |
| any event after `INTIME + 24h` | **hidden** (windowed out in SQL) | not available at prediction time |
| measurement counts | **altered**: capped to the 24h window | full-stay counts encode stay length |
| `DOB` / age | **altered**: clipped to 90 | MIMIC shifts >89y DOB ~300y |
| `CHARTEVENTS.ERROR = 1` | **hidden** (dropped) | clinician-flagged bad values |
| `HOSPITAL_EXPIRE_FLAG`, `DISCHARGE_LOCATION`, `DEATHTIME` | **hidden** | known only at/after discharge |
| `ICUSTAYS.LOS` precomputed | **hidden** | direct copy of the target |

Imputation/scaling are done *inside* the model pipeline (Phase 4) so test folds never leak in.

### 3.2 Build feature matrices (without / with labs) + missingness indicators
Two matrices on identical rows so the labs ablation is apples-to-apples; `add_missing_indicators`
appends the informative `*_measured` columns from 2.4.

In [ ]:
with profiler.track("2_feature_engineering"):
    fm_base = build_feature_matrix(data, include_labs=False, add_missing_indicators=True)
    fm      = build_feature_matrix(data, include_labs=True,  add_missing_indicators=True)
print(f"without labs : {fm_base.X.shape[1]} features")
print(f"with labs    : {fm.X.shape[1]} features  (+{fm.X.shape[1]-fm_base.X.shape[1]} lab cols)")
print("rows identical:", fm.X.index.equals(fm_base.X.index))

### 3.3 Feature inventory -- how many attributes and of which categories

In [ ]:
cats = categorize_features(fm.feature_names)
display(cats.to_frame())
fig = plots.feature_category_bar(cats); plots.save(fig, "feature_categories.png"); plt.show()
print("Post-engineering missingness in the kept matrix (imputed later, in-pipeline):")
fig = plots.missingness_bar(fm.X.isna().mean(), top=25); plots.save(fig, "missingness_matrix.png"); plt.show()

### 3.4 Unsupervised structure exploration (PCA / t-SNE / clustering)
*Exploratory only -- this does NOT predict LOS.* We ask: does the first-24h feature space have
structure, and does it align with stay length? Honest expectation on clinical data: weak
separation (low silhouette); t-SNE is a 2-D visualization, not a distance-true map.

In [ ]:
with profiler.track("3_unsupervised"):
    pca = uns.pca_view(fm.X, n_components=10)
    tsne_xy, tsne_idx = uns.tsne_view(fm.X)
    clus = uns.kmeans_silhouette(fm.X, k_range=range(2, 7))

fig = plots.variance_scree(pca.explained); plots.save(fig, "pca_scree.png"); plt.show()
fig = plots.embedding_scatter(pca.coords, fm.y_reg.values, "PCA (coloured by LOS days)")
plots.save(fig, "pca_scatter.png"); plt.show()
fig = plots.embedding_scatter(tsne_xy, fm.y_clf.loc[tsne_idx].values,
                              "t-SNE (coloured by LOS bucket)", "LOS bucket", categorical=True)
plots.save(fig, "tsne_scatter.png"); plt.show()
fig = plots.silhouette_plot(clus.k_scores); plots.save(fig, "silhouette.png"); plt.show()

print(f"best k = {clus.best_k} (silhouette {clus.k_scores[clus.best_k]:.3f} -- "
      f"{'weak' if clus.k_scores[clus.best_k] < 0.3 else 'moderate'} structure)")
print("\nCluster LOS profile (do phenotypes differ in stay length?):")
display(uns.cluster_profile(clus.labels, fm.y_reg, fm.y_clf))
print("PC1/PC2 top loadings:")
display(pca.loadings.abs().sum(axis=1).sort_values(ascending=False).head(8).to_frame("|loading|"))

### 3.5 Train / test split -- grouped by SUBJECT_ID

In [ ]:
X_tr, X_te, y_tr, y_te, g_tr, g_te = grouped_train_test_split(fm.X, fm.y_reg, fm.groups)
assert_no_group_leakage(g_tr, g_te)
XB_tr, XB_te = fm_base.X.loc[X_tr.index], fm_base.X.loc[X_te.index]
CODE = {"short": 0, "medium": 1, "long": 2}; INV = {v: k for k, v in CODE.items()}
yc = fm.y_clf.map(CODE); yc_tr, yc_te = yc.loc[y_tr.index], yc.loc[y_te.index]
print(f"train {X_tr.shape[0]} stays / {g_tr.nunique()} patients | "
      f"test {X_te.shape[0]} stays / {g_te.nunique()} patients (no overlap)")

## Phase 4 -- Modeling

### 4.1 Published baselines + SOTA proposal
We anchor against the literature, not just our own trivial baselines.

* **Trivial baselines** (mean / majority) -- a model must beat these; a baseline R^2 ~ 0 is
  evidence the target is not leaked.
* **Published baselines** (table below) -- Harutyunyan et al. 2019 (linear kappa ~0.34, LSTM
  kappa ~0.43), MIMIC-Extract (LOS>7d AUROC ~0.76), first-24h regression (R^2 ~0.04).
* **Proposed SOTA for this tabular setting:** gradient-boosted trees (XGBoost / LightGBM),
  which match or beat deep nets on medium tabular data (Grinsztajn et al. 2022). The only DL
  worth trying would operate on the raw hourly *sequence* (GRU/temporal CNN), which we leave as
  future work -- on LOS specifically its gains over trees are modest.

In [ ]:
display(literature_baselines())

### 4.2 Regression -- model comparison + LABEVENTS ablation (GroupKFold)
`improvement` = MAE reduction (days) from adding labs.

In [ ]:
with profiler.track("4_regression_cv_ablation"):
    reg_delta, reg_base, reg_labs = harness.ablation_table(
        registry.build_regressors, XB_tr, X_tr, y_tr, g_tr, "regression")
print("MAE (days) without vs with labs:"); display(reg_delta.round(3))
display(reg_labs[["MAE_mean", "RMSE_mean", "R2_mean", "fit_time_s"]].round(3))
fig = plots.model_comparison(reg_labs, "MAE_mean", "Regression CV (with labs) -- MAE",
                             lower_is_better=True); plots.save(fig, "reg_compare.png"); plt.show()

### 4.3 Regression -- hyper-parameter search (RandomizedSearchCV, GroupKFold)

In [ ]:
with profiler.track("5_hp_search"):
    tuned_name, tuned_model, best_params, cv_mae = harness.tune_regressor(X_tr, y_tr, g_tr)
print(f"Tuned {tuned_name}: CV MAE = {cv_mae:.3f} d\n{best_params}")

### 4.4 Classification -- model comparison + LABEVENTS ablation

In [ ]:
with profiler.track("6_classification_cv_ablation"):
    clf_delta, clf_base, clf_labs = harness.ablation_table(
        registry.build_classifiers, XB_tr, X_tr, yc_tr, g_tr, "classification")
print("quadratic kappa without vs with labs:"); display(clf_delta.round(3))
display(clf_labs[["accuracy_mean","f1_macro_mean","kappa_quadratic_mean","roc_auc_ovr_mean"]].round(3))
fig = plots.model_comparison(clf_labs, "kappa_quadratic_mean",
                             "Classification CV (with labs) -- quadratic kappa",
                             lower_is_better=False); plots.save(fig, "clf_compare.png"); plt.show()

## Phase 5 -- Evaluation

### 5.1 Regression hold-out: quantify the lab contribution, vs published baselines

In [ ]:
with profiler.track("7_holdout_eval"):
    yp_labs, _, _ = harness.holdout_fit_predict(tuned_model, X_tr, y_tr, X_te)
    yp_base, _, _ = harness.holdout_fit_predict(clone(tuned_model), XB_tr, y_tr, XB_te)
m_labs, m_base = metrics.regression_metrics(y_te, yp_labs), metrics.regression_metrics(y_te, yp_base)
cmp = pd.DataFrame({"without_labs": m_base, "with_labs": m_labs}).T
display(cmp[["MAE","RMSE","R2","within_1d"]].round(3))
print(f"Lab lift on test: MAE {m_base['MAE']:.3f} -> {m_labs['MAE']:.3f} d, "
      f"R2 {m_base['R2']:.3f} -> {m_labs['R2']:.3f}")
lit = literature_baselines()
print("\nvs literature (regression):")
display(lit[lit.framing == "regression"])
fig = plots.predicted_vs_actual(y_te, yp_labs, f"{tuned_name} (with labs)")
plots.save(fig, "pred_vs_actual.png"); plt.show()
print("\nError by true-LOS band (the failure mode -- long stays under-predicted):")
display(metrics.error_by_los_range(y_te, yp_labs).round(2))

### 5.2 Feature importance + distributions of the most important variables

In [ ]:
est = tuned_model.named_steps["model"]
top_feats = []
if hasattr(est, "feature_importances_"):
    imp = pd.Series(est.feature_importances_, index=fm.feature_names)
    imp = imp / imp.sum() if imp.sum() else imp
    fig = plots.feature_importance(list(fm.feature_names), est.feature_importances_, 20)
    plots.save(fig, "feat_importance.png"); plt.show()
    print(f"Lab features = {imp[[i for i in imp.index if i.startswith('lab_')]].sum():.1%} of importance; "
          f"missingness indicators = {imp[[i for i in imp.index if i.endswith('_measured')]].sum():.1%}")
    # distributions of the most important CONTINUOUS features, by LOS bucket
    cont = [i for i in imp.sort_values(ascending=False).index
            if any(i.endswith(s) for s in ('_mean','_min','_max','_std')) or i == 'age_years']
    top_feats = cont[:9]
if top_feats:
    fig = plots.feature_distributions_by_class(fm.X, fm.y_clf, top_feats)
    plots.save(fig, "feat_distributions.png"); plt.show()

### 5.3 Classification hold-out + confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
best_clf_name = clf_labs.index[0]; best_clf = registry.build_classifiers()[best_clf_name]
yc_pred, yc_proba, _ = harness.holdout_fit_predict(best_clf, X_tr, yc_tr, X_te, want_proba=True)
labels = [0, 1, 2]
print(f"HOLD-OUT classification -- {best_clf_name} (with labs)")
for k, v in metrics.classification_metrics(yc_te, yc_pred, yc_proba, labels=labels).items():
    print(f"  {k:16s}: {v:.3f}")
print("\n", classification_report(yc_te.map(INV),
      pd.Series(yc_pred, index=yc_te.index).map(INV), zero_division=0))
cm = confusion_matrix(yc_te, yc_pred, labels=labels)
fig = plots.confusion(cm, [INV[i] for i in labels]); plots.save(fig, "confusion.png"); plt.show()
print("vs literature: Harutyunyan LSTM quadratic kappa ~0.43 (ordinal LOS).")

## Phase 6 -- Deployment considerations

### 6.1 Profiling -- total time and per-phase breakdown

In [ ]:
report = profiler.report(); display(report)
fig = plots.profiling(report); plots.save(fig, "profiling.png"); plt.show()
print(f"TOTAL execution time: {profiler.total:.1f}s")

### 6.2 Big data: did we use MapReduce? (and why / why not)

**We did not hand-write Hadoop/Spark MapReduce -- on purpose.** The workload is
*filter-then-aggregate* over two large tables (CHARTEVENTS ~330M rows, LABEVENTS ~27M). The
right tool is **BigQuery**, whose engine (Dremel) executes exactly this as a massively-parallel,
distributed **map-reduce-style** plan under the hood: the `WHERE`/`JOIN` is the *map* (run on
thousands of shards), the `GROUP BY ... AVG/MIN/MAX/COUNT` is the *reduce*. We push that down in
SQL ([`src/data/sql.py`](src/data/sql.py)) and only a few-hundred-thousand aggregated rows ever
leave the warehouse.

Writing a bespoke Spark/Hadoop MapReduce job would reproduce what BigQuery already does, add
cluster-management overhead, and be slower to develop -- *over-engineering*. So MapReduce is
present **conceptually and via the engine**, just not as boilerplate we maintain. If the data
lived in flat files with no warehouse, Spark would be the natural choice and the same
map (parse+filter+window) / reduce (aggregate per stay-concept) decomposition would apply.

### 6.3 Multiprocessing -- where parallelism is actually used

* **Server-side (BigQuery):** the aggregation fans out across many workers (above).
* **Cross-validation & model fitting:** scikit-learn's `n_jobs=-1` (`cfg.N_JOBS`) uses
  **joblib** to run GroupKFold folds and RandomForest trees across all CPU cores in parallel
  ([`src/evaluation/harness.py`](src/evaluation/harness.py), [`src/models/registry.py`](src/models/registry.py)).
* **Boosting libraries:** XGBoost/LightGBM use their own multithreaded (`n_jobs`) histogram
  builders within each fit.
* **Caching as amortization:** BigQuery results are parquet-cached so repeated runs skip the
  distributed step entirely.

We deliberately do **not** multiprocess the feature-engineering pandas code: after aggregation
the matrix is small (~10^5 x ~150) and vectorised pandas/NumPy is already memory-bound, so
process-pool overhead would only slow it down.

## Conclusions (fill bracketed values from a BigQuery run)

* **Labs help, in the right direction and where expected.** Hold-out MAE moved by [..] d and
  quadratic kappa by [..]; labs were [..]% of importance, concentrated in tree models (the
  linear model barely benefits -- labs add non-linear organ-dysfunction signal). Informative-
  missingness indicators contributed [..]%.
* **Regression stays hard** (R^2 ~ [..], MAE ~ [..] d) but clearly beats the mean baseline and
  is in line with the first-24h literature; **classification** (quadratic kappa ~ [..]) is
  comparable to the Harutyunyan LSTM benchmark while using interpretable trees.
* **Structure is weak** (best silhouette [..]): patients do not split into clean phenotypes, so
  a supervised model is the right call -- the unsupervised views are descriptive, not predictive.
* **Big-data handled by pushing map-reduce-style aggregation into BigQuery**; total runtime
  [..]s on the compact local matrix.

### References
1. Wirth & Hipp (2000). CRISP-DM 1.0.
2. Johnson et al. (2016). MIMIC-III. *Scientific Data*.
3. Harutyunyan et al. (2019). Multitask learning & benchmarking with clinical time series.
4. Wang et al. (2020). MIMIC-Extract. *ACM CHIL*.
5. Grinsztajn et al. (2022). Why tree-based models still outperform DL on tabular data. *NeurIPS*.
6. Sharafoddini et al. (2019). Patient phenotyping / informative missingness in MIMIC.
7. Dean & Ghemawat (2008). MapReduce. *CACM*; Melnik et al. (2010). Dremel (BigQuery engine).